# Entrenament model offline - Predicció glucosa

El codi, el preprocessament i l'entrenament s'han fet tenint en compte la exploració de dades del dataset realitzant previament, per tal d'entrenar els models de forma clara i estructurada.

#### Import de les llibreries necessaries

In [101]:
import pandas as pd
import numpy as np

from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error


import warnings

warnings.simplefilter("ignore", FutureWarning)

pd.set_option('display.max_columns', None)

#### Definició de les columnes dels datasets

In [102]:
cols = [
    'year', 'month', 'day', 'hour', 'minute', 'second', # A–F
    'glucose_level', # G
    'finger_stick', # H
    'basal', # I
    'bolus', # J
    'sleep', # K
    'work', # L
    'stressors', # M
    'hypo_event', # N
    'illness', # O
    'exercise', # P
    'basis_heart_rate', # Q
    'basis_gsr', # R
    'basis_skin_temperature', # S
    'basis_air_temperature',  # T
    'basis_step', # U
    'basis_sleep', # V
    'meal', # W
    'meal_type' # X
]

#### Definició de pacients i horitzons

In [103]:
# Definicio de numero dels pacients
PACIENTS = [559, 563, 570, 575, 588, 591]

# Definició de l'horitzo
HORITZO = {30:6, 60:12}  # minuts : files (pas de 5 mins)

## Carrega del dataset

In [104]:
# Funcio per obtenir els datasets dels pacients
def load_data(pacient, train_or_test):
    df=pd.read_csv(f'../data/{pacient}/{pacient}_{train_or_test}.csv', sep=';', header = None, names = cols)
    return df

## Preprocessament del dataset

In [ ]:
def preprocess(df):
    prep = df.copy()

    # Afegim un time per unificar la dada temporal
    prep['time'] = pd.to_datetime(dict(year=df.year, month=df.month, day=df.day, hour=df.hour, minute=df.minute))
    
    # Ens asegurem que estiguin ordenades de forma cronologica
    prep.sort_values('time', inplace=True)

    # Coma decimal a punt
    # He observat que aqueste categories estan mal asignades com a object, farem el canvi de coma a punt perque detacti el decimal
    # Convertirem les columnes a numeriques
    convert = ["basal","bolus","basis_gsr","basis_skin_temperature","basis_air_temperature"]
    
    for c in convert:
        prep[c] = (prep[c].astype(str)
                   .str.replace(",",".", regex=False)
                   .str.strip()
                   .astype(float))

    # Unifiquem el tipo de meal que han fet
    cat_meal = {
        1:"Desayuno",
        2:"Almuerzo",
        3:"Cena",
        4:"Snack",
        5:"Correccion_hipo"
    }

    prep["meal_type"] = prep["meal_type"].map(cat_meal).astype("category")
    prep = pd.get_dummies(prep, columns=['meal_type'], dummy_na=False, prefix='meal')


    
    
    # Zeros que no poden ser valids
    invalid_zero = [
        "glucose_level",
        "basis_heart_rate",
        "basis_gsr",
        "basis_skin_temperature",
        "basis_air_temperature"
    ]
    
    prep[invalid_zero] = prep[invalid_zero].replace(0, np.nan)

    # Fem servir forward-fill per tal d'asegurarnos que no es mira al futur
    # Limitem el ffill a maxim 30 mins (6 files) de nan consecutius.
    prep['glucose_level'] = prep['glucose_level'].fillna(method='ffill', limit=6) 

    # ffill sense limits ja que ens dona millors resultats en la predicció
    prep[invalid_zero] = prep[invalid_zero].fillna(method='ffill') 
    prep = prep.dropna(subset=['glucose_level'])
    
    # Tornem a treure la columna time ja que no son valors entrenables
    prep = prep.drop(columns=['time'])
    return prep

## Entrenament del model

#### Definició de la funcio de split features i target

In [106]:
def make_xy(df, files):

    # Definim y com al nivell de glucosa a predir
    # Agafarem el valor a pedir "x" files més amunt segons l'horitzo (30 mins: 6 files o 60 min: 12 files)
    y = df['glucose_level'].shift(-files)

    # Definim X amb totes les columnes pero sense les ultimes files, ja que no tindran predicció y
    X = df.iloc[:-files].copy()

    # Ajustem la y perque tingui el mateix nombre de files que x (eliminant les ultimes files ja que no poden ser predites)
    y = y.iloc[:-files]

    return X, y

#### Definició de la funció de evaluació del model

In [107]:
def evaluate(y_true, y_pred):
    # Calculem el rmse i mae donat el y_true i el y_pred (valor que ha predit el model vs el real)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mae  = mean_absolute_error(y_true, y_pred)
    return rmse, mae

### Entrenament per cada pacient (model offline)

In [108]:
resultats = []

for pacient in PACIENTS:
    print(f'\nPacient {pacient}')

    # Cridem la funció load_data per carregar les dades sense processar.
    train_raw = load_data(pacient, 'train')
    test_raw  = load_data(pacient, 'test')

    # Preprocessem les dades tal i com hem definit anteriorment, per tal que no hi hagi dades buides i fer una bona imputació
    # per tal que el model "no miri al futur"
    train = preprocess(train_raw)
    test  = preprocess(test_raw)

    # Igualem les columnes del train y del test i emplenem el que falti amb 0
    # He observat que el test no te la categoria de corrección hipo i per tant te una columna menys.
    # La mantindrem en el train ja que pot significar una millor deteccio de la glucosa.
    train, test = train.align(test, join='outer', axis=1, fill_value=0)

    # Ignorem els 60 minuts primers del test ja que ni podrien ser predits i es podria correlacionar amb les ultimes files del train
    test = test.iloc[12:].reset_index(drop=True)

    for minuts, files in HORITZO.items():

        # Entrenament del model
        X_train, y_train = make_xy(train, files)

        model = RandomForestRegressor(
            n_estimators=1000,
            max_depth=10,
            min_samples_leaf=5,
            random_state=42,
            n_jobs=-1
        )
        model.fit(X_train, y_train)


        # Predicció del test
        # Treiem les ultimes files segons l'horitzo, ja que no podran ser predites
        X_test = test.iloc[:-files].copy()

        # El true del test, serà el desplaçament cap amunt de les files.
        y_true = test['glucose_level'].shift(-files)

        # Finalment també treiem les files que no han pogut ser predites menors a l'horitzo
        y_true = y_true.iloc[:-files].reset_index(drop=True)

        # Fem la predicció del model que hem entrenat
        y_pred = model.predict(X_test)



        # guardem la predicció a csv
        directori_pred = f'../data/predicted/pred{pacient}_{minuts}min.csv'

        # Fem index més 12 per tal de poder comparar les prediccions visualment amb més facilitat (anteriorment hem tret 1 hora del test)
        df_pred = pd.DataFrame({'Index': X_test.index + 6, f'pred_glucosa_t+{minuts}': y_pred}) # Comença als 30 mins
        df_pred.to_csv(directori_pred, index=False)

            

        # Evaluació del model
        # Cridem la funció que hem definit anteriorment de evaluació que ens retorna rmse i mae
        rmse, mae = evaluate(y_true, y_pred)

        # Afegim els restats en un diccionari per posteriorment poder-los mostrar
        resultats.append({
            'Pacient': pacient,
            'Horitzo': minuts,
            'RMSE': rmse,
            'MAE' : mae
        })


        print(f'{minuts} min: RMSE={rmse:.2f}  MAE={mae:.2f}')




Pacient 559
30 min: RMSE=24.36  MAE=16.86
60 min: RMSE=38.12  MAE=27.95

Pacient 563
30 min: RMSE=20.84  MAE=15.28
60 min: RMSE=33.63  MAE=24.80

Pacient 570
30 min: RMSE=18.67  MAE=13.21
60 min: RMSE=31.23  MAE=23.27

Pacient 575
30 min: RMSE=24.32  MAE=18.06
60 min: RMSE=40.71  MAE=32.07

Pacient 588
30 min: RMSE=21.82  MAE=15.88
60 min: RMSE=33.36  MAE=24.54

Pacient 591
30 min: RMSE=25.33  MAE=19.05
60 min: RMSE=39.20  MAE=31.86


## Taula final de resutats

In [109]:
resultats_df = pd.DataFrame(resultats)

taula = (resultats_df.pivot(index='Pacient', columns='Horitzo', values=['RMSE','MAE']))
print(f'Visualització inicial de la taula: \n{taula}')

# Renombrem les columnes de la taula
taula.columns = ['RMSE 30','RMSE 60','MAE 30', 'MAE 60']

# Reordenem les columnes
taula = taula[['RMSE 30','MAE 30','RMSE 60','MAE 60']]

# Calculem el promig
promig = taula.mean().to_frame().T
promig.index = ['PROMIG'] 

# Mostrem la taula final amb el promig
taula_final = pd.concat([taula, promig], axis=0)

print("\nRESULTATS FINALS:")
print(taula_final)

Visualització inicial de la taula: 
              RMSE                   MAE           
Horitzo         30         60         30         60
Pacient                                            
559      24.359533  38.120418  16.856433  27.951652
563      20.835878  33.634172  15.278918  24.797058
570      18.673662  31.226884  13.208677  23.267981
575      24.315632  40.712588  18.055355  32.073523
588      21.823414  33.363818  15.877730  24.536662
591      25.330601  39.198965  19.054134  31.856394

RESULTATS FINALS:
          RMSE 30     MAE 30    RMSE 60     MAE 60
559     24.359533  16.856433  38.120418  27.951652
563     20.835878  15.278918  33.634172  24.797058
570     18.673662  13.208677  31.226884  23.267981
575     24.315632  18.055355  40.712588  32.073523
588     21.823414  15.877730  33.363818  24.536662
591     25.330601  19.054134  39.198965  31.856394
PROMIG  22.556453  16.388541  36.042807  27.413878
